# Kapitel 5 koduppgifter

### 8. 

In [1]:
import numpy as np
from sklearn.decomposition import PCA
# Creating a dataset with 3 features/columns
X = np.random.rand(1000, 3)
print(X[0:5])
# Reducing the data to 2 dimensions
pca = PCA(n_components=2)
X2D = pca.fit_transform(X)
print(X2D[0:5])
# "Recreating" the data to 3 dimensions
X3D_inv = pca.inverse_transform(X2D)
# Not exactly equal since some information was lost in the transformation
print(np.allclose(X3D_inv, X))

[[0.97158209 0.43643767 0.6297593 ]
 [0.01300785 0.68696047 0.38986597]
 [0.59606604 0.96636272 0.43905559]
 [0.19859936 0.69485119 0.22078946]
 [0.51361796 0.24483582 0.02156777]]
[[ 0.45401763  0.14985386]
 [-0.47596807 -0.13851446]
 [ 0.10979729 -0.0253396 ]
 [-0.27805338 -0.29253531]
 [ 0.0375189  -0.5031367 ]]
False


### 9.

In [2]:
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

df = pd.read_csv("data/car_price_dataset.csv", sep=";")
print(df.shape)
df.head()

(10000, 10)


,Brand,Model,Year,Engine_Size,Fuel_Type,Transmission,Mileage,Doors,Owner_Count,Price
0,Kia,Rio,2020,4.2,Diesel,Manual,289944,3,5,8501
1,Chevrolet,Malibu,2012,2.0,Hybrid,Automatic,5356,2,3,12092
2,Mercedes,GLA,2020,4.2,Diesel,Automatic,231440,4,2,11171
3,Audi,Q5,2023,2.0,Electric,Manual,160971,2,1,11780
4,Volkswagen,Golf,2003,2.6,Hybrid,Semi-Automatic,286618,3,3,2867


In [3]:
# Kategorisk data one-hot-encodas, sedan split
df = pd.get_dummies(df, columns=["Brand", "Model", "Fuel_Type", "Transmission"],
                    drop_first=True)

X = df.drop(columns=["Price"])
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# PCA kräver skalade variabler, annars dominerar Mileage helt
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Antal variabler efter one-hot:", X.shape[1])

Antal variabler efter one-hot: 48


In [4]:
# Utan PCA
start = time.time()
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_s, y_train)
tid_utan = time.time() - start

pred = rf.predict(X_test_s)
rmse_utan = root_mean_squared_error(y_test, pred)
r2_utan = r2_score(y_test, pred)

print(f"Utan PCA: RMSE {rmse_utan:.2f}, R2 {r2_utan:.4f}, tid {tid_utan:.1f} s")

Utan PCA: RMSE 544.92, R2 0.9677, tid 0.4 s


In [5]:
# Med PCA, behåll 95 procent av variansen
pca = PCA(n_components=0.95, random_state=42)
X_train_p = pca.fit_transform(X_train_s)
X_test_p = pca.transform(X_test_s)

print("Antal komponenter kvar:", pca.n_components_, "av", X.shape[1])

start = time.time()
rf_pca = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_pca.fit(X_train_p, y_train)
tid_med = time.time() - start

pred_pca = rf_pca.predict(X_test_p)
rmse_med = root_mean_squared_error(y_test, pred_pca)
r2_med = r2_score(y_test, pred_pca)

print(f"Med PCA:  RMSE {rmse_med:.2f}, R2 {r2_med:.4f}, tid {tid_med:.1f} s")

Antal komponenter kvar: 35 av 48
Med PCA:  RMSE 937.60, R2 0.9043, tid 1.2 s


In [6]:
print("RMSE utan PCA:", round(rmse_utan, 2), " med PCA:", round(rmse_med, 2))
print("R2   utan PCA:", round(r2_utan, 4), " med PCA:", round(r2_med, 4))
print("Tid  utan PCA:", round(tid_utan, 1), "s  med PCA:", round(tid_med, 1), "s")

RMSE utan PCA: 544.92  med PCA: 937.6
R2   utan PCA: 0.9677  med PCA: 0.9043
Tid  utan PCA: 0.4 s  med PCA: 1.2 s


Med PCA blev modellen sämre (större fel i priset) och långsammare, trots att den hade färre kolumner (35 i stället för 48).

Varför blev den sämre?
- PCA vet inte att det är priset du vill gissa. Den behåller det som skiljer bilarna mest åt, inte det som påverkar priset mest.
- Viktiga saker som årsmodell och miltal blandas ihop med mindre viktiga saker, som bilmärke, till nya kolumner.
- Random forest fungerar genom att ställa enkla frågor, som "Är bilen från efter 2015?". När allt är ihopblandat går sådana frågor inte att ställa, och modellen blir sämre.

Varför blev den långsammare?
- Många av de ursprungliga kolumnerna är bara ja/nej, till exempel "Är det en Audi?". Sådana går snabbt att använda.
- Kolumnerna som PCA gör är vanliga decimaltal, och de tar längre tid att räkna med.

Slutsats: PCA hjälper mest när det finns många kolumner som säger ungefär samma sak, till exempel pixlar i bilder. Här gjorde den mer skada än nytta.